In [2]:
import os
import shutil
import numpy as np
import torch
from tqdm import tqdm

# ================= 配置区域 =================
DATASET_ROOT = "./dataset_tri_RAD_Split"
SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")

CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK", 
    "16QAM", "32QAM", "64QAM", 
    "2ASK", "16APSK", "32APSK", "FSK2"
]

FILE_MAPPING = {
    "E:/Project/AMR-Benchmark/PSK8-03-24-25-10h54m48s14.iq": "8PSK",
    "E:/Project/AMR-Benchmark/QAM16-03-24-25-10h52m09s762.iq": "16QAM",
    "E:/Project/AMR-Benchmark/QAM64-03-24-25-10h51m51s805.iq": "64QAM",
    "E:/Project/AMR-Benchmark/ASK2-03-24-25-10h56m09s501.iq": "2ASK",
    "E:/Project/AMR-Benchmark/BPSK-03-24-25-10h52m32s937.iq": "BPSK",
    "E:/Project/AMR-Benchmark/QPSK-03-24-25-10h53m20s46.iq": "QPSK",
    "E:/Project/AMR-Benchmark/FSK2DEV625KHZ-03-24-25-11h08m50s462.iq": "FSK2",
    "E:/Project/AMR-Benchmark/OQPSK-03-24-25-10h54m02s6.iq": "OQPSK",
    "E:/Project/AMR-Benchmark/PI4DQPSK-03-24-25-10h54m23s982.iq": "PI4DQPSK"
}

SAMPLES_PER_IMAGE = 10240
IMAGE_SIZE = 64
TOTAL_SIM_SAMPLES_PER_CLASS = 800
TRAIN_VAL_RATIO = 0.8
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# ================= 工具函数 =================
def fast_hist2d(x, y, bins=64, range_bounds=((-1, 1), (-1, 1))):
    x_min, x_max = range_bounds[0]
    y_min, y_max = range_bounds[1]
    x = torch.clamp(x, x_min, x_max)
    y = torch.clamp(y, y_min, y_max)
    x_idx = ((x - x_min) / (x_max - x_min) * (bins - 1)).long()
    y_idx = ((y - y_min) / (y_max - y_min) * (bins - 1)).long()
    flat_idx = y_idx * bins + x_idx
    hist = torch.bincount(flat_idx, minlength=bins*bins)
    return hist.view(bins, bins).float()

def generate_tensor_data(iq_complex_data, img_size=64):
    if iq_complex_data.dim() == 1:
        iq_complex_data = iq_complex_data.unsqueeze(0)
    iq_data = iq_complex_data.squeeze()
    if iq_data.numel() < 100: return torch.zeros(3, img_size, img_size)

    scale = torch.sqrt(torch.mean(iq_data.real**2 + iq_data.imag**2)) + 1e-9
    norm_iq = iq_data / (3.0 * scale)
    
    real, imag = norm_iq.real, norm_iq.imag
    phase, mag = torch.angle(norm_iq) / np.pi, torch.abs(norm_iq)
    
    diff_sig = norm_iq[1:] * torch.conj(norm_iq[:-1])
    d_scale = torch.sqrt(torch.mean(diff_sig.real**2 + diff_sig.imag**2)) + 1e-9
    diff_sig = diff_sig / (3.0 * d_scale)
    d_real, d_imag = diff_sig.real, diff_sig.imag
    
    bins = img_size
    hist_r = fast_hist2d(real, imag, bins=bins, range_bounds=((-1.2, 1.2), (-1.2, 1.2)))
    hist_a = fast_hist2d(phase, mag, bins=bins, range_bounds=((-1.0, 1.0), (0.0, 1.5)))
    hist_d = fast_hist2d(d_real, d_imag, bins=bins, range_bounds=((-1.2, 1.2), (-1.2, 1.2)))
    
    def process(h):
        h = torch.log1p(h)
        mx = h.max()
        if mx > 1e-6: h = h / mx
        return h

    tensor = torch.stack([process(hist_r), process(hist_a), process(hist_d)], dim=0)
    return torch.flip(tensor, [1])

# 尝试导入仿真函数
try:
    from data_generator_rad import generate_sim_signal
except ImportError:
    print("Error: Could not import generate_sim_signal from data_generator_rad.py")
    exit()

# ================= 主执行逻辑 =================
if __name__ == "__main__":
    if os.path.exists(DATASET_ROOT): 
        print(f"Cleaning {DATASET_ROOT}...")
        shutil.rmtree(DATASET_ROOT)
    
    print("===== 1. Generating Sim Data (Split-Ready) =====")
    for cls_name in CLASS_NAMES:
        os.makedirs(os.path.join(SIM_TRAIN_DIR, cls_name), exist_ok=True)
        os.makedirs(os.path.join(SIM_VAL_DIR, cls_name), exist_ok=True)
        # 建立 Real Test 的子文件夹 [关键]
        os.makedirs(os.path.join(REAL_TEST_DIR, cls_name), exist_ok=True)
        
        print(f"Generating {cls_name}...")
        for i in tqdm(range(TOTAL_SIM_SAMPLES_PER_CLASS)):
            target_dir = SIM_TRAIN_DIR if np.random.rand() < TRAIN_VAL_RATIO else SIM_VAL_DIR
            target_dir = os.path.join(target_dir, cls_name)
            
            snr = np.random.uniform(-5, 25)
            sig = generate_sim_signal(cls_name, SAMPLES_PER_IMAGE, snr)
            tensor_data = generate_tensor_data(sig.to(DEVICE), IMAGE_SIZE)
            torch.save(tensor_data.cpu(), os.path.join(target_dir, f"{cls_name}_{i:04d}_{snr:.1f}.pt"))

    print("\n===== 2. Processing Real Data (CRITICAL) =====")
    for file_path, label in FILE_MAPPING.items():
        if not os.path.exists(file_path): 
            print(f"Skipping (Not Found): {file_path}")
            continue
            
        print(f"Processing Real File: {label}...")
        # 确保保存路径是 dataset_tri_RAD_Split/real_test/CLASS_NAME/
        save_dir = os.path.join(REAL_TEST_DIR, label)
        os.makedirs(save_dir, exist_ok=True)
        
        try:
            raw = np.fromfile(file_path, dtype=np.int16)
            if len(raw) % 2 != 0: raw = raw[:-1]
            # I/Q 解析
            I = raw[::2].astype(np.float32)
            Q = raw[1::2].astype(np.float32)
            iq_complex = (I + 1j * Q).astype(np.complex64) / 32767.0
            
            num_slices = len(iq_complex) // SAMPLES_PER_IMAGE
            print(f" -> Found {num_slices} samples.")
            
            for i in range(num_slices):
                chunk = iq_complex[i*SAMPLES_PER_IMAGE : (i+1)*SAMPLES_PER_IMAGE]
                chunk_tensor = torch.from_numpy(chunk).to(DEVICE)
                
                # 生成 Tensor
                tensor_data = generate_tensor_data(chunk_tensor, IMAGE_SIZE)
                
                # 保存
                save_name = f"real_{i:04d}.pt"
                torch.save(tensor_data.cpu(), os.path.join(save_dir, save_name))
                
        except Exception as e:
            print(f"Error processing {label}: {e}")

    print("\nDone! Dataset structure check:")
    print(f"Real Test dir exists? {os.path.exists(REAL_TEST_DIR)}")
    print(f"Sample BPSK dir exists? {os.path.exists(os.path.join(REAL_TEST_DIR, 'BPSK'))}")

Cleaning ./dataset_tri_RAD_Split...
===== 1. Generating Sim Data (Split-Ready) =====
Generating BPSK...


100%|██████████| 800/800 [00:03<00:00, 222.97it/s]


Generating QPSK...


100%|██████████| 800/800 [00:02<00:00, 305.67it/s]


Generating 8PSK...


100%|██████████| 800/800 [00:03<00:00, 245.41it/s]


... (15 outputs left out)

100%|██████████| 800/800 [00:03<00:00, 244.73it/s]


Generating FSK2...


100%|██████████| 800/800 [00:02<00:00, 320.67it/s]



===== 2. Processing Real Data (CRITICAL) =====
Processing Real File: 8PSK...
 -> Found 4882 samples.
Processing Real File: 16QAM...
 -> Found 4882 samples.
Processing Real File: 64QAM...
 -> Found 4882 samples.
Processing Real File: 2ASK...
 -> Found 4882 samples.
Processing Real File: BPSK...
 -> Found 4882 samples.
Processing Real File: QPSK...
 -> Found 4882 samples.
Processing Real File: FSK2...
 -> Found 4882 samples.
Processing Real File: OQPSK...
 -> Found 4882 samples.
Processing Real File: PI4DQPSK...
 -> Found 4882 samples.

Done! Dataset structure check:
Real Test dir exists? True
Sample BPSK dir exists? True
